# Chapter 2 — NB 07: regressão do burden ZNF175 × tinnitus, Park pipeline em v1 e v2

**O veredito.** Pipeline **idêntico** nas duas coortes (filtro do Park, NB 06):
`tinnitus ~ carrier + age + age² + sex + PC1–10`, logística; + tabela 2×2 / Fisher (robusto p/ contagens pequenas);
+ estratificado EUR/AFR com meta-análise IVW (modelo do Park).

**Tinnitus (regra do 2):** caso = ICD `388.3x` (ICD-9) ou `H93.1x` (ICD-10) em ≥2 datas distintas; controle = nunca; 1 data = excluído.

**Montagem (multi-fonte):**
- v1: carriers(GENO_ID) + Demographics(GENO_ID↔PT_ID, sexo, BIRTH_YEAR) + evec_gia(PC1–10, GIA) + Diagnosis(PT_ID→tinnitus). age = 2020 − BIRTH_YEAR.
- v2: carriers(PMBB_ID) + covariates(sex, Age_at_Enrollment, PC1–10) + exome_ancestries(Class) + icd.txt(tinnitus).

In [1]:
from pathlib import Path
import pandas as pd, numpy as np
import statsmodels.api as sm
from scipy.stats import fisher_exact, norm

BASE=Path("/project/hall/analysis/hearing-loss-genomics")
R06=BASE/"analysis/chapter_2/results/06"
R07=BASE/"analysis/chapter_2/results/07"; R07.mkdir(parents=True, exist_ok=True)
FZ="/static/PMBB/PMBB_Freeze17"
V2P=BASE/"data/pmbb_v2/Phenotype/2.0"
TIN=r'^(388\.3|H93\.1)'   # tinnitus ICD-9 388.3x / ICD-10 H93.1x
PCS=[f"PC{i}" for i in range(1,11)]

## 1. Montar v2 (Hui)

In [2]:
carr2 = pd.read_csv(R06/"carriers_v2.csv").rename(columns={"IID":"PMBB_ID"})
cov2  = pd.read_csv(V2P/"PMBB-Release-2020-2.0_phenotype_covariates.txt", sep="\t")
anc2  = pd.read_csv("/static/PMBB/PMBB-Release-2020-2.0/Exome/PCA/PMBB-Release-2020-2.0_genetic_exome_ancestries.txt",
                    sep="\t", usecols=["PMBB_ID","Class"])
icd2  = pd.read_csv(V2P/"PMBB-Release-2020-2.0_phenotype_icd.txt", sep="\t",
                    usecols=["PMBB_ID","CODE","ENC_DATE_SHIFT"], dtype=str)
tin2  = icd2[icd2["CODE"].str.match(TIN, na=False)]
nd2   = tin2.groupby("PMBB_ID")["ENC_DATE_SHIFT"].nunique()

d2 = (carr2.merge(cov2, on="PMBB_ID").merge(anc2, on="PMBB_ID", how="left"))
d2["tin_dates"] = d2["PMBB_ID"].map(nd2).fillna(0)
d2 = d2[d2["tin_dates"] != 1].copy()
d2["tinnitus"] = (d2["tin_dates"] >= 2).astype(int)
d2["age"] = pd.to_numeric(d2["Age_at_Enrollment"], errors="coerce")
d2["sex_male"] = (d2["sex"]=="Male").astype(int)
d2["anc"] = d2["Class"]
print("v2 N:", len(d2), "| carriers:", int(d2.carrier.sum()),
      "| tinnitus cases:", int(d2.tinnitus.sum()))

v2 N: 42614 | carriers: 69 | tinnitus cases: 841


## 2. Montar v1 (Park)

In [3]:
carr1 = pd.read_csv(R06/"carriers_v1.csv").rename(columns={"IID":"GENO_ID"})
demo  = pd.read_csv(f"{FZ}/phenotype/PMBB_Geno_Demographics_Deidentified_012020.csv")
demo1 = demo.drop_duplicates("GENO_ID")[["GENO_ID","PT_ID","GENDER_CODE","BIRTH_YEAR"]]
evec  = pd.read_csv(f"{FZ}/genotype/imputed/Additional_Files/OMNI_GSA_V1_V2_evec_gia.txt", sep=r"\s+")
# evec IID uses dashes (UPENN-...), carriers/demographics use underscores -> match on stable UPENN##### token
evec1 = evec[["IID"]+PCS+["GIA"]].copy()
evec1["tok"] = evec1["IID"].str.extract(r"(UPENN\d+)")
evec1 = evec1.dropna(subset=["tok"]).drop_duplicates("tok")
diag  = pd.read_csv(f"{FZ}/phenotype/PMBB_Geno_Nonsensitive_Diagnosis_Deidentified_012020.csv",
                    header=None, usecols=[0,1,6], names=["PT_ID","CODE","DATE"], dtype=str)
tin1  = diag[diag["CODE"].str.match(TIN, na=False)]
nd1   = tin1.groupby("PT_ID")["DATE"].nunique()

d1 = carr1.merge(demo1, on="GENO_ID")
d1["tok"] = d1["GENO_ID"].str.extract(r"(UPENN\d+)")
d1 = d1.merge(evec1[["tok"]+PCS+["GIA"]], on="tok", how="left")
d1["PT_ID"] = d1["PT_ID"].astype(str)
nd1.index = nd1.index.astype(str)
d1["tin_dates"] = d1["PT_ID"].map(nd1).fillna(0)
d1 = d1[d1["tin_dates"] != 1].copy()
d1["tinnitus"] = (d1["tin_dates"] >= 2).astype(int)
d1["age"] = 2020 - pd.to_numeric(d1["BIRTH_YEAR"], errors="coerce")
d1["sex_male"] = (d1["GENDER_CODE"]=="M").astype(int)
d1["anc"] = d1["GIA"]
print("v1 N:", len(d1), "| carriers:", int(d1.carrier.sum()),
      "| tinnitus cases:", int(d1.tinnitus.sum()))

v1 N: 9161 | carriers: 26 | tinnitus cases: 131


## 3. Enriquecimento bruto (2×2 + Fisher) — robusto para contagens pequenas

In [4]:
def two_by_two(d, label):
    ct = pd.crosstab(d["carrier"], d["tinnitus"])
    # garantir 2x2
    for r in (0,1):
        if r not in ct.index: ct.loc[r]=[0,0]
    for c in (0,1):
        if c not in ct.columns: ct[c]=0
    ct = ct.sort_index().reindex(columns=[0,1])
    a,b,c,dd = ct.loc[1,1], ct.loc[1,0], ct.loc[0,1], ct.loc[0,0]  # carrier-case, carrier-ctrl, noncar-case, noncar-ctrl
    orr,p = fisher_exact([[a,b],[c,dd]])
    rate_car = a/(a+b) if (a+b)>0 else np.nan
    rate_non = c/(c+dd) if (c+dd)>0 else np.nan
    print(f"[{label}] carriers: {a} tinnitus / {a+b} ({rate_car*100:.1f}%) | "
          f"non-carriers: {c} / {c+dd} ({rate_non*100:.2f}%) | Fisher OR={orr:.2f} p={p:.2e}")
    return dict(label=label, car_case=int(a), car_n=int(a+b), non_case=int(c), non_n=int(c+dd),
                fisher_or=orr, fisher_p=p)

r2 = two_by_two(d2, "v2 (Hui, 44k)")
r1 = two_by_two(d1, "v1 (Park, 11k)")

[v2 (Hui, 44k)] carriers: 4 tinnitus / 69 (5.8%) | non-carriers: 837 / 42545 (1.97%) | Fisher OR=3.07 p=4.76e-02
[v1 (Park, 11k)] carriers: 4 tinnitus / 26 (15.4%) | non-carriers: 127 / 9135 (1.39%) | Fisher OR=12.90 p=4.68e-04


## 4. Regressão logística ajustada (pooled, PC1–10)

In [5]:
def run_logit(d, label):
    dd = d.dropna(subset=["carrier","age","sex_male","tinnitus"]+PCS).copy()
    dd["age2"] = dd["age"]**2
    X = sm.add_constant(dd[["carrier","age","age2","sex_male"]+PCS])
    y = dd["tinnitus"]
    try:
        res = sm.Logit(y, X).fit(disp=0, maxiter=200)
        b, se, p = res.params["carrier"], res.bse["carrier"], res.pvalues["carrier"]
        print(f"[{label}] N={len(dd)} cases={int(y.sum())} | carrier beta={b:.3f} OR={np.exp(b):.2f} "
              f"SE={se:.3f} p={p:.2e}")
        return dict(label=label, n=len(dd), cases=int(y.sum()), beta=b, se=se, p=p)
    except Exception as e:
        print(f"[{label}] logit failed: {e}")
        return dict(label=label, n=len(dd), cases=int(y.sum()), beta=np.nan, se=np.nan, p=np.nan)

L2 = run_logit(d2, "v2 (Hui, 44k)")
L1 = run_logit(d1, "v1 (Park, 11k)")

[v2 (Hui, 44k)] N=42605 cases=841 | carrier beta=1.262 OR=3.53 SE=0.520 p=1.52e-02


[v1 (Park, 11k)] N=9161 cases=131 | carrier beta=2.679 OR=14.58 SE=0.586 p=4.79e-06


## 5. Estratificado EUR/AFR + meta-análise IVW (modelo do Park)

In [6]:
def strat_meta(d, label):
    rows=[]
    for anc in ["EUR","AFR"]:
        sub = d[d["anc"]==anc]
        r = run_logit(sub, f"{label} {anc}")
        rows.append(r)
    # IVW meta dos coef de carrier (onde válido)
    valid=[r for r in rows if np.isfinite(r["beta"]) and np.isfinite(r["se"]) and r["se"]>0]
    if valid:
        w=[1/r["se"]**2 for r in valid]
        bm=sum(r["beta"]*wi for r,wi in zip(valid,w))/sum(w)
        sem=np.sqrt(1/sum(w)); z=bm/sem; pm=2*(1-norm.cdf(abs(z)))
        print(f"[{label}] META carrier OR={np.exp(bm):.2f} p={pm:.2e}  (strata: {[r['label'].split()[-1] for r in valid]})")
        return dict(label=label, meta_or=np.exp(bm), meta_p=pm)
    print(f"[{label}] meta: sem estratos válidos (poucos casos)"); return dict(label=label, meta_or=np.nan, meta_p=np.nan)

M1 = strat_meta(d1, "v1 (Park)")
M2 = strat_meta(d2, "v2 (Hui)")

[v1 (Park) EUR] N=6632 cases=76 | carrier beta=3.518 OR=33.71 SE=0.627 p=2.05e-08


[v1 (Park) AFR] N=1754 cases=38 | carrier beta=-20.764 OR=0.00 SE=100158.069 p=1.00e+00
[v1 (Park)] META carrier OR=33.71 p=2.05e-08  (strata: ['EUR', 'AFR'])


[v2 (Hui) EUR] N=29518 cases=571 | carrier beta=1.524 OR=4.59 SE=0.526 p=3.74e-03


/project/hall/analysis/hearing-loss-genomics/venv/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


[v2 (Hui) AFR] N=10925 cases=242 | carrier beta=-21.854 OR=0.00 SE=136740.314 p=1.00e+00
[v2 (Hui)] META carrier OR=4.59 p=3.74e-03  (strata: ['EUR', 'AFR'])


/project/hall/analysis/hearing-loss-genomics/venv/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


## 6. Interpretação — a "história dos casos" (não confundir as 3 quantidades)

| | N total | **Tinnitus (total)** | **Portadores ZNF175** (pLOF raro) | **Portador E tinnitus** |
|---|---|---|---|---|
| v1 (11k) | 9.161 | **131** (1,4%) | **26** (0,28%) | **4** |
| v2 (44k) | 42.614 | **841** (2,0%) | **69** (0,16%) | **4** |

- **Tinnitus é COMUM** (centenas de casos); ser **portador** das variantes pLOF do ZNF175 é **raro**. O burden testa se a taxa de tinnitus é **maior entre portadores** que no resto.
- v1: **4/26** portadores (15,4%) têm tinnitus vs **1,4%** de fundo → OR ~13.
- v2: **4/69** portadores (5,8%) vs **2,0%** → OR ~3,5.
- O nº de **portadores-com-tinnitus ficou em ~4** nas duas coortes, enquanto o **pool de portadores cresceu** (26→69). Os portadores novos do v2 são ultra-raros **sem** tinnitus → **diluem** o enriquecimento → o OR regride. **Sinal ancorado em poucos casos = winner's curse.**

In [7]:
# --- Summary ---
summary = f'''# NB 07 — ZNF175 burden x tinnitus, Park pipeline on both cohorts

Model: tinnitus(rule-of-2, ICD 388.3x/H93.1x) ~ carrier + age + age2 + sex + PC1-10. Same pipeline v1 & v2.

## Cohort sizes (after rule-of-2 exclusion)
- v1 (Park, Freeze One): N={len(d1)}, carriers={int(d1.carrier.sum())}, tinnitus cases={int(d1.tinnitus.sum())}
- v2 (Hui, Release 2.0): N={len(d2)}, carriers={int(d2.carrier.sum())}, tinnitus cases={int(d2.tinnitus.sum())}

## Carrier-vs-tinnitus enrichment (2x2 / Fisher)
- v1: carriers {r1["car_case"]}/{r1["car_n"]} tinnitus vs non {r1["non_case"]}/{r1["non_n"]} | OR={r1["fisher_or"]:.2f} p={r1["fisher_p"]:.2e}
- v2: carriers {r2["car_case"]}/{r2["car_n"]} tinnitus vs non {r2["non_case"]}/{r2["non_n"]} | OR={r2["fisher_or"]:.2f} p={r2["fisher_p"]:.2e}

## Adjusted logistic (pooled, PC1-10)
- v1: OR={np.exp(L1["beta"]):.2f} p={L1["p"]:.2e} (N={L1["n"]}, cases={L1["cases"]})
- v2: OR={np.exp(L2["beta"]):.2f} p={L2["p"]:.2e} (N={L2["n"]}, cases={L2["cases"]})

## Stratified EUR/AFR + IVW meta
- v1: meta OR={M1["meta_or"]:.2f} p={M1["meta_p"]:.2e}
- v2: meta OR={M2["meta_or"]:.2f} p={M2["meta_p"]:.2e}

## Verdict (to interpret)
Same Park pipeline on both: if the carrier-tinnitus signal is present in v1 (11k) but absent in v2 (44k),
the loss is winner's curse / dilution (real null at scale), NOT a pipeline artifact. Small carrier/case counts
-> Fisher is the robust readout; logistic/meta are sensitivity checks.

## Outputs
- d1/d2 analysis frames saved as v1_analysis.csv / v2_analysis.csv
'''
d1.to_csv(R07/"v1_analysis.csv", index=False); d2.to_csv(R07/"v2_analysis.csv", index=False)
(R07/"nb07_regression_summary.md").write_text(summary)
print(summary)

# NB 07 — ZNF175 burden x tinnitus, Park pipeline on both cohorts

Model: tinnitus(rule-of-2, ICD 388.3x/H93.1x) ~ carrier + age + age2 + sex + PC1-10. Same pipeline v1 & v2.

## Cohort sizes (after rule-of-2 exclusion)
- v1 (Park, Freeze One): N=9161, carriers=26, tinnitus cases=131
- v2 (Hui, Release 2.0): N=42614, carriers=69, tinnitus cases=841

## Carrier-vs-tinnitus enrichment (2x2 / Fisher)
- v1: carriers 4/26 tinnitus vs non 127/9135 | OR=12.90 p=4.68e-04
- v2: carriers 4/69 tinnitus vs non 837/42545 | OR=3.07 p=4.76e-02

## Adjusted logistic (pooled, PC1-10)
- v1: OR=14.58 p=4.79e-06 (N=9161, cases=131)
- v2: OR=3.53 p=1.52e-02 (N=42605, cases=841)

## Stratified EUR/AFR + IVW meta
- v1: meta OR=33.71 p=2.05e-08
- v2: meta OR=4.59 p=3.74e-03

## Verdict (to interpret)
Same Park pipeline on both: if the carrier-tinnitus signal is present in v1 (11k) but absent in v2 (44k),
the loss is winner's curse / dilution (real null at scale), NOT a pipeline artifact. Small carrier/case co